<a href="https://colab.research.google.com/github/aahan-charak24/Deep-Learning-Mastery/blob/main/Codes/YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print(torch.cuda.is_available())

False


<h1>Define YOLO backbone </h1>

# 🧠 Simplified YOLO Model (PyTorch)

This is a simplified YOLO-style object detection model.  
It predicts bounding boxes and class probabilities over a grid.

---

## 🔹 Model Structure

The model has two main parts:

- **Backbone** → extracts features from the image  
- **Detection Head** → produces final predictions  

---

## 🔹 1. Backbone (Feature Extractor)

```python
self.backbone = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2)
)

```
📌 What it does:
Input: (B, 3, H, W)
Extracts features using convolutions
Reduces spatial size using stride + pooling
📌 Output:
(B, 128, H/4, W/4)

🔹 2. Detection Head
``` python
self.detections = nn.Sequential(
    nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(256),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(
        256 * (grid_size // 4)**2,
        grid_size * grid_size * (num_anchors * 5 + num_classes)
    )
)
```
📌 Steps:
Conv → increases channels (128 → 256)
Flatten → converts feature map into vector
Linear → outputs predictions for all grid cells

🔹 3. Output Representation

Each grid cell predicts:

num_anchors * 5 + num_classes

Where:

5 → (x, y, w, h, confidence)
num_classes → class probabilities

🔹 4. Forward Pass
``` python
def forward(self, X):
    feat = self.backbone(X)
    predictions = self.detections(feat)

    return predictions.view(
        -1,
        self.grid_size,
        self.grid_size,
        self.num_anchors * 5 + self.num_classes
    )
    ```
📌 Flow:
Image → backbone → feature map
Feature map → detection head → predictions
Reshape → grid format

🔹 5. Final Output Shape
(B, S, S, num_anchors * 5 + num_classes)

Where:

B = batch size
S = grid size
🔹 6. Intuition
The image is divided into an S × S grid
Each grid cell predicts:
Multiple bounding boxes (anchors)
Object confidence
Class probabilities
🔹 7. Example

If:

grid_size = 7
num_anchors = 2
num_classes = 20

Then:

Output per cell = 2 * 5 + 20 = 30
Final shape = (B, 7, 7, 30)

In [ ]:
import torch.nn as nn

class YOLO_T(nn.Module):
  def __init__(self, num_classes, num_anchors, grid_size = 7):
    super().__init__()
    self.num_classes = num_classes
    self.num_anchors = num_anchors
    self.grid_size = grid_size
    #backbone for extracting the features
    self.backbone = nn.Sequential(
      nn.Conv2d(3, 64, kernel_size = 7, stride = 2, padding = 3),
      nn.BatchNorm2d(64),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size = 2, stride = 2),
      nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
      nn.BatchNorm2d(128),
      nn.ReLU(),
      nn.MaxPool2d(kernel_size = 2, stride = 2)
    )

    #detection heads
    self.detections = nn.Sequential(
        nn.Conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.Flatten(),
        #first part before comma is input size to the linear layer, second is the number of predictions
        nn.Linear(256 * (grid_size // 4)**2, grid_size * grid_size * (num_anchors * 5 + num_classes))
    )


  #forward pass
  def forward(self, X):
    feat = self.backbone(X)
    predictions = self.detections(feat)
    #view (-1) automatically gives us the batch size in this case. so for each grid in the image, we output the bounding boxes and probability etc.

    return predictions.view(-1, self.grid_size, self.grid_size, self.num_anchors * 5 + self.num_classes)

In [ ]:
m1 = YOLO_T(10, 3)
print(m1)

YOLO_T(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (detections): Sequential(
    (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Flatten(start_dim=1, end_dim=-1)
    (4): Linear(in_features=256, out_features=1225, bias=True)
  )
)


In [ ]:
del(m1)

<h1>Making the Code more modular </h1>

Image b →
    Grid S×S →
        Each cell →
            vector of size A*(5+C)

<h1>Defining anchor boxes </h1>


Anchors = cookie cutters
 Model = reshapes them to fit objects

<h1> Dataset prep to YOLO format </h1>
box is at pixel 150 nope ❌, box center is at 37.5% of image width ✅
So, center point is relative to width and height and the box_width and height are relative to image width and height

<h1>Data Augmentation </h1<>

<h1>Loss Functions </h1>


*   Localization Loss: Penalizes inaccurate bounding box predictions.
*   Confidence Loss: Measures how confident the model is about the presence of
an object.
*   Classification Loss: Compares predicted classes with ground truth.


lambda_coord and lambda_noobj?

They are just weights (importance multipliers) in the loss function.

They tell the model:

“This type of error is more important, so punish it more.”

<br><br> YOLO is a regression problem

YOLO predicts:

box coordinates (x, y, w, h)
confidence
class probabilities

Everything is treated as continuous values

So squared error fits naturally.

<h1>Loading Data using DataLoader</h1>

<h1>Training Loop </h1>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import VOCDetection
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import xml.etree.ElementTree as ET
import numpy as np
from torch.utils.data import Dataset

<h1>Transforming the dataset </h1>

In [3]:
transform = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor()
])

In [4]:
dataset = VOCDetection(
    root = "./data",
    year = "2012",
    image_set = "train",
    download = True,
    transform=transform
)

small_dataset = torch.utils.data.Subset(dataset, range(500))

100%|██████████| 2.00G/2.00G [01:12<00:00, 27.5MB/s]


<h1>Convert bbox format suitable to yolo</h1>

In [5]:
def convertBbox(size, box):
  #width and height of the image
  w, h = size

  xmin, ymin, xmax, ymax = box
  #centroid relative
  x = (xmin + xmax) / 2.0 / w
  y = (ymin + ymax) / 2.0 / h
  #relative width and height of bounding box
  bw = (xmax - xmin) / w
  bh = (ymax - ymin) / h

  return x, y , bw, bh



In [6]:
VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle',
    'bus','car','cat','chair','cow',
    'diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor'
]

classes_to_index = {cls: i for i, cls in enumerate(VOC_CLASSES)}

print(f"Classes are: \n {classes_to_index}")

Classes are: 
 {'aeroplane': 0, 'bicycle': 1, 'bird': 2, 'boat': 3, 'bottle': 4, 'bus': 5, 'car': 6, 'cat': 7, 'chair': 8, 'cow': 9, 'diningtable': 10, 'dog': 11, 'horse': 12, 'motorbike': 13, 'person': 14, 'pottedplant': 15, 'sheep': 16, 'sofa': 17, 'train': 18, 'tvmonitor': 19}


<h1>Target tensor, for this we are only using one anchor per cell </h1>

S.S.30

In [18]:
S = 7
B = 2
C = 20
# S cell size, B: no of anchors per cell, C number of classes

def create_target(annotation, img_size):
  target = torch.zeros((S, S, 5 + C))

  w, h = img_size
  #loop through annotations
  for obj in annotation["object"]:
    #extract classname and bounding box
    cls = obj["name"]
    bbox = obj["bndbox"]
    #extract the co-ordinates
    xmin = int(bbox['xmin'])
    xmax = int(bbox['xmax'])
    ymin = int(bbox['ymin'])
    ymax = int(bbox['ymax'])

    #convert to bounding box format for YOLO
    x, y, bw, bh = convertBbox((w, h), (xmin, ymin, xmax, ymax))

    #find out the centroid belongs to which cell
    i = int(x * S)
    j = int(y * S)

    class_idx = classes_to_index[cls]

    #create target
    target[j, i, 0:5] = torch.tensor([x, y, bw, bh, 1])
    target[j, i, 5 + class_idx] = 1


  return target




<h1>Dataset</h1>

In [19]:
class YOLODataset(Dataset):
  def __init__(self, dataset):
    self.dataset = dataset

  def __len__(self):
    return len(self.dataset)

  def __getitem__(self, i):
    img, target = self.dataset[i]

    annotation = target['annotation']
    size = annotation['size']

    w, h = int(size['width']), int(size['height'])


    #create target format suitable for yolo
    yolo_target = create_target(annotation=annotation, img_size = (w, h))

    return img, yolo_target



<h1>DataLoader </h1>

In [20]:
train_dataset = YOLODataset(small_dataset)

train_loader = DataLoader(train_dataset, batch_size =64, shuffle = True)

<h1>Simple YOLO architecture </h1>

In [26]:

class YOLO(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1),  # 224
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), # 112
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), # 56
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, 2, 1), # 28
            nn.ReLU(),
        )

        # FORCE 7x7
        self.pool = nn.AdaptiveAvgPool2d((S, S))

        self.head = nn.Conv2d(256, 1*5 + C, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.head(x)

        x = x.permute(0, 2, 3, 1)  # (B, 7, 7, 30)
        return x

In [27]:
criterion = nn.MSELoss()

model = YOLO()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
print(model)

YOLO(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): ReLU()
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (7): ReLU()
  )
  (pool): AdaptiveAvgPool2d(output_size=(7, 7))
  (head): Conv2d(256, 25, kernel_size=(1, 1), stride=(1, 1))
)


In [28]:
from tqdm import tqdm

In [29]:
n_epochs = 10
model.train()

for epoch in range(n_epochs):

  tot_loss = 0.0
  for imgs, target in tqdm(train_loader):
    optimizer.zero_grad()
    #forward pass
    preds = model(imgs)
    loss = criterion(preds, target)
    #backprop
    loss.backward()
    optimizer.step()

    tot_loss += loss.item()

  print(f'Epoch {epoch+1}, loss: {np.mean(tot_loss)}')








100%|██████████| 8/8 [01:36<00:00, 12.06s/it]


Epoch 1, loss: 0.056172532960772514


100%|██████████| 8/8 [01:32<00:00, 11.57s/it]


Epoch 2, loss: 0.05329486168920994


100%|██████████| 8/8 [01:39<00:00, 12.46s/it]


...

Epoch 8, loss: 0.05251462943851948


100%|██████████| 8/8 [01:31<00:00, 11.42s/it]


Epoch 9, loss: 0.052402700297534466


100%|██████████| 8/8 [01:48<00:00, 13.57s/it]

Epoch 10, loss: 0.052189840003848076
